# 07 - Sentiment Uncertainty

**Learning objective:** Refine a binary entity-sentiment prompt and use log probabilities across five independent runs to explore average positivity and variation.

This extends [06 - Entity Sentiment](06_entity_sentiment.ipynb), but includes its own configuration, article loading, and reviewed entity inventory so it runs from a fresh kernel.

**Prerequisites:** Install `openai`, `pydantic` (version 2 or newer), and `tiktoken` in this kernel (for example, `%pip install openai 'pydantic>=2' tiktoken`). Set `OPENAI_API_KEY` or replace the API-key placeholder. Use `gpt-4.1`, a workshop model that supports structured outputs, temperature sampling, and `logprobs` / `top_logprobs`.

**Your task:** Edit the `TODO` binary sentiment prompt and interpret the results. Imports, parallel execution, probability extraction, and aggregation are complete. Run cells in order. Next: [08 - Function Calling](08_function_calling.ipynb).

In [ ]:
import os

API_KEY = os.environ.get("OPENAI_API_KEY", "insert key here")
MODEL = "gpt-4.1"  # Workshop model with logprobs and temperature support.

In [ ]:
import json
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from statistics import fmean, stdev
from typing import Literal

import tiktoken
from openai import OpenAI
from pydantic import BaseModel, Field


def make_client():
    if not API_KEY or API_KEY == "insert key here":
        raise ValueError("Set OPENAI_API_KEY or replace the API_KEY placeholder.")
    return OpenAI(api_key=API_KEY, timeout=120.0, max_retries=0)


client = make_client()

## Step 1: Load the Article

This cell is complete and needs no edits. It locates `data/text/article.txt` from this notebook's directory or the repository root. The article is fictional; all judgments concern its portrayal of the named companies.

In [ ]:
article_path = next(
    (parent / "data" / "text" / "article.txt"
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "text" / "article.txt").is_file()),
    None,
)
if article_path is None:
    raise FileNotFoundError(
        "Cannot find data/text/article.txt. Start the kernel inside this repository."
    )
doc_article = article_path.read_text(encoding="utf-8")
print(doc_article)

## Step 2: Establish a Consistent Entity List

The supplied inventory has been reviewed against this article and contains its 15 named companies, with aliases merged. Compare it with your Exercise 1 result. Run this cell once before the experiment and keep the inventory fixed across all five iterations. Rediscovering names in each run would make omissions and aliases look like sentiment variation.

The evidence schema and all settings below are provided. Each run produces one short evidence assessment for every exact inventory name.

In [ ]:
# Reviewed inventory for the supplied article; aliases are merged once, here.
ENTITY_NAMES = (
    "Carrefour", "DHL Group", "Microsoft", "Amazon",
    "Schneider Electric", "Siemens", "Maersk", "SAP", "Unilever",
    "Tesco", "Procter & Gamble", "Nestlé", "IBM", "Walmart", "Alibaba",
)
N_RUNS = 5
EVIDENCE_TEMPERATURE = 0.8
SCORE_TEMPERATURE = 1.0
print(json.dumps(ENTITY_NAMES, indent=2, ensure_ascii=False))


class EntityEvidence(BaseModel):
    entity: str = Field(description="Exact company name from the supplied inventory.")
    assessment: str = Field(
        description="Two short sentences weighing article evidence for this company."
    )


class EvidenceAnalysis(BaseModel):
    evidence: list[EntityEvidence] = Field(
        description="Exactly one evidence assessment for each supplied company."
    )

## Step 3: Write Your Binary Sentiment Prompt

Replace the starter below with your policy for judging sentiment toward each target company. Use only `positive` and `negative`; `neutral` is unavailable in this exercise.

- Define favorable and unfavorable evidence, keeping the focus on the target company.
- State how to weigh mixed evidence, realized outcomes, and speculation.
- Decide explicitly how to force a choice when evidence is balanced, merely descriptive, or insufficient. What meaning will a negative label have under your rule?

The same prompt is used for evidence assessment and scoring. The scaffolding adds stage-specific output instructions automatically. Your work is prompt design, not changing the probability or threading code.

In [ ]:
# TODO: Replace this starter with your binary entity-sentiment policy.
# TODO: Explain how mixed and insufficient evidence become positive or negative.
BINARY_SENTIMENT_PROMPT = """
Assess whether the article presents each target company positively or negatively.
""".strip()

## Step 4: Understand the Sampling and Scoring

Each of **exactly five independent iterations** first samples a short evidence assessment for all companies at `temperature=0.8`, then scores each company separately at `temperature=1.0`. Temperature permits sampling variation. Each request uses a fresh conversation, and no results are passed between iterations.

**Why sample evidence first?** Sampling only a final label can vary the label while leaving the underlying probabilities for a fixed input unchanged. Here the sampled assessment varies the scoring context. Score variation therefore explores sensitivity to sampled readings as well as any serving nondeterminism. Five runs may still agree.

The exact lowercase labels `positive` and `negative` are verified below as distinct single tokens for `MODEL`. The scoring response has exactly one token. Equal +100 `logit_bias` values encourage these labels without favoring one relative to the other; returned log probabilities belong to this biased, temperature-controlled request. We request the top 20 alternatives at that same position and compute:

`positive_score = P(positive) / (P(positive) + P(negative))`

Subtracting the larger log probability before exponentiating avoids numerical underflow in the normalization. We never use the probability of the entire generated response as a sentiment score. If an alternative is missing, non-finite, or marked unavailable, the code stops with an explicit error; it never substitutes zero or silently averages fewer than five scores. All code below is complete.

In [ ]:
LABEL_TO_SENTIMENT = {"positive": "positive", "negative": "negative"}
encoding = tiktoken.encoding_for_model(MODEL)
label_token_ids = {}
for label in LABEL_TO_SENTIMENT:
    token_ids = encoding.encode(label)
    if len(token_ids) != 1 or encoding.decode(token_ids) != label:
        raise ValueError(f"{label!r} is not one exact token for {MODEL}.")
    label_token_ids[label] = token_ids[0]
if len(set(label_token_ids.values())) != 2:
    raise ValueError("The two labels must have different token IDs.")
LABEL_BIAS = {str(token_id): 100 for token_id in label_token_ids.values()}
print({label: {"sentiment": LABEL_TO_SENTIMENT[label], "token_id": token_id}
       for label, token_id in label_token_ids.items()})


class ScoreUnavailableError(RuntimeError):
    pass


def extract_positive_score(choice):
    if choice.message.refusal:
        raise ScoreUnavailableError(f"Scoring refused: {choice.message.refusal}")
    token_infos = choice.logprobs.content if choice.logprobs else None
    if not token_infos or len(token_infos) != 1:
        raise ScoreUnavailableError("Expected log probabilities for one label token.")
    token_info = token_infos[0]
    if (choice.message.content not in LABEL_TO_SENTIMENT
            or token_info.token != choice.message.content):
        raise ScoreUnavailableError("Expected exactly positive or negative, with no whitespace.")

    # Use the sampled token too: it need not be repeated in top_logprobs.
    logps = {item.token: item.logprob for item in token_info.top_logprobs}
    logps[token_info.token] = token_info.logprob
    missing = set(LABEL_TO_SENTIMENT) - logps.keys()
    if missing:
        raise ScoreUnavailableError(
            f"Alternative-label log probability unavailable for {sorted(missing)}; "
            "no score computed. Review the prompt/model; do not substitute zero."
        )
    a, b = logps["positive"], logps["negative"]
    # The API can use -9999.0 for an unavailable log probability.
    if any(not math.isfinite(value) or value <= -9999.0 for value in (a, b)):
        raise ScoreUnavailableError("Non-finite or sentinel label log probability.")
    offset = max(a, b)
    positive_weight = math.exp(a - offset)
    negative_weight = math.exp(b - offset)
    score = positive_weight / (positive_weight + negative_weight)
    return score, LABEL_TO_SENTIMENT[token_info.token]

### One Independent Iteration: Supplied Implementation

This function checks that the sampled evidence covers the fixed inventory exactly once, then scores each company using the article and its assessment. `max_tokens=1` intentionally ends after the label; a `length` finish reason is normal for these scoring calls. Raw assessments and sampled labels are retained so you can investigate score differences.

In [ ]:
def run_iteration(run_id):
    with make_client() as run_client:
        evidence_completion = run_client.beta.chat.completions.parse(
            model=MODEL,
            temperature=EVIDENCE_TEMPERATURE,
            messages=[
                {"role": "system", "content": BINARY_SENTIMENT_PROMPT + (
                    "\nPrepare evidence assessments, not final labels. For each supplied "
                    "company, weigh favorable and unfavorable passages in two short "
                    "sentences. Note missing evidence. Preserve every supplied name "
                    "exactly once and add no companies."
                )},
                {"role": "user", "content": (
                    f"Companies: {json.dumps(ENTITY_NAMES, ensure_ascii=False)}\n\n"
                    f"Article:\n{doc_article}"
                )},
            ],
            response_format=EvidenceAnalysis,
        )
        message = evidence_completion.choices[0].message
        if message.refusal or message.parsed is None:
            raise RuntimeError(f"No parsed evidence returned: {message.refusal}")
        evidence_rows = message.parsed.evidence
        evidence_by_entity = {row.entity: row.assessment for row in evidence_rows}
        if (len(evidence_rows) != len(ENTITY_NAMES)
                or set(evidence_by_entity) != set(ENTITY_NAMES)
                or any(not text.strip() for text in evidence_by_entity.values())):
            raise ValueError("Evidence must cover the fixed inventory exactly once.")

        rows = []
        for entity in ENTITY_NAMES:
            score_completion = run_client.chat.completions.create(
                model=MODEL,
                temperature=SCORE_TEMPERATURE,
                max_tokens=1,
                logprobs=True,
                top_logprobs=20,
                logit_bias=LABEL_BIAS,
                messages=[
                    {"role": "system", "content": BINARY_SENTIMENT_PROMPT + (
                        "\nOutput exactly one lowercase word: positive or negative."
                        " No explanation, punctuation, whitespace, or neutral label."
                    )},
                    {"role": "user", "content": (
                        f"Target company: {entity}\n\nArticle:\n{doc_article}\n\n"
                        f"Sampled evidence assessment:\n{evidence_by_entity[entity]}"
                    )},
                ],
            )
            try:
                score, sentiment = extract_positive_score(score_completion.choices[0])
            except ScoreUnavailableError as exc:
                raise ScoreUnavailableError(f"Run {run_id}, {entity}: {exc}") from exc
            rows.append({
                "run": run_id, "entity": entity, "sentiment": sentiment,
                "positive_score": score, "assessment": evidence_by_entity[entity],
            })
        return rows

## Step 5: Execute Exactly Five Runs in Parallel

Run this cell once per prompt experiment. Five workers each process the full inventory, with companies scored sequentially inside a worker. There are at most five concurrent requests. One evidence request plus 15 scoring requests per iteration gives **80 API requests** for the experiment. SDK retries are disabled; errors are surfaced rather than adding runs. Re-running this cell starts a new five-run experiment.

In [ ]:
run_results = []
with ThreadPoolExecutor(max_workers=N_RUNS) as executor:
    futures = {executor.submit(run_iteration, run_id): run_id
               for run_id in range(1, N_RUNS + 1)}
    for future in as_completed(futures):
        run_id = futures[future]
        try:
            run_results.extend(future.result())
        except Exception as exc:
            raise RuntimeError(
                f"Iteration {run_id} failed; no five-run summary is valid. {exc}"
            ) from exc
run_results.sort(key=lambda row: (row["run"], ENTITY_NAMES.index(row["entity"])))
print(f"Completed {N_RUNS} iterations for {len(ENTITY_NAMES)} companies.")

## Step 6: Aggregate and Display

This complete cell checks that all 15 companies have five scores, prints the individual scores, and displays the required summary. `stdev` computes the **sample** standard deviation, dividing by `5 - 1 = 4`. Aggregation uses full-precision scores; only display values are rounded. Inspect `run_results` to compare the sampled assessments and labels behind the scores.

In [ ]:
expected_keys = {(run_id, entity) for run_id in range(1, N_RUNS + 1)
                 for entity in ENTITY_NAMES}
actual_keys = [(row["run"], row["entity"]) for row in run_results]
if len(actual_keys) != len(expected_keys) or set(actual_keys) != expected_keys:
    raise ValueError("Need exactly five complete, aligned runs; refusing partial data.")
if any(not math.isfinite(row["positive_score"])
       or not 0.0 <= row["positive_score"] <= 1.0 for row in run_results):
    raise ValueError("Every positivity score must be finite and between zero and one.")

summary = []
for entity in ENTITY_NAMES:
    scores = [row["positive_score"] for row in run_results if row["entity"] == entity]
    summary.append({
        "entity": entity,
        "mean_positive_score": fmean(scores),
        "positive_score_std": stdev(scores),  # Sample standard deviation: denominator n - 1.
    })

print("Five per-run positivity scores for each company:")
for entity in ENTITY_NAMES:
    scores = [row["positive_score"] for row in run_results if row["entity"] == entity]
    print(f"{entity}: " + ", ".join(f"{score:.6f}" for score in scores))
print("\nEntity-level summary (full precision retained in summary):")
print(json.dumps([
    {key: round(value, 6) if isinstance(value, float) else value
     for key, value in row.items()} for row in summary
], indent=2, ensure_ascii=False))

## Step 7: Interpret and Refine

- A higher `mean_positive_score` indicates more positive sentiment **under this scoring method**.
- A larger `positive_score_std` indicates greater variation in the positivity scores across the five runs. It is not the fraction of labels that disagree.
- These are **exploratory indicators of model confidence and consistency, not calibrated guarantees of correctness or reliability**. They depend on the prompt, model, label encoding, label bias, temperature, sampled evidence, and article. Five samples provide only a small exploratory view.
- Very low variation is a valid result. Six-decimal rounding can hide small differences; the unrounded values remain available in `summary` and `run_results`.

**Reflect:**
1. Which companies have competing evidence, and which simply lack enough information? How does your forced-choice rule affect their mean scores?
2. For a company whose scores vary, what changed in the sampled assessments? Did different sampled labels necessarily mean different probabilities?
3. Refine one binary-prompt instruction and re-run the prompt, five-run execution, and aggregation cells. Did the mean shift more than the within-experiment variation?
4. Could a consistently biased prompt produce a low standard deviation? Why would neither a high positivity score nor low variation prove the analysis correct?

**Expected outcome:** An entity-level summary containing `entity`, `mean_positive_score`, and `positive_score_std`, plus a short interpretation of how prompt wording shaped the results. Continue with [08 - Function Calling](08_function_calling.ipynb).